# Thermal defect classifier — train + test (YOLO26x + CV)
**Just: Runtime → Change runtime type → GPU, then Runtime → Run all.**

Only requirement: your dataset must be on Google Drive. On your Mac run
```bash
cd /Volumes/dronisight
zip -r transformer.zip yolo_thermal_transformer
```
and upload `transformer.zip` anywhere in **My Drive** (or one folder deep). This notebook auto-finds it — no paths to edit.

**Architecture:** one YOLO26x detector localizes the **transformer**; hot conductors/connections and hotspots are then found by **CV (relative heat)** inside the transformer crop — there is **no wire model** (thin clustered conductors weren't learnable; CV directly targets 'where is it hot').

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab pins: 8.3.x silently degrades yolo26 -> nano; pillow 11.3+ is broken.
!pip install -q -U "ultralytics>=8.4.60" "pillow==11.2.1" "scikit-image"
!git clone -q https://github.com/arupa444/transformer_train.git /content/transformer_train
import sys; sys.path.insert(0, '/content/transformer_train/src')
print('ok')

In [ ]:
# Auto-find the dataset on Drive (a 'transformer.zip' or a 'yolo_thermal_transformer'
# folder), in My Drive or one folder deep. No path editing needed.
import glob, os, zipfile, shutil, yaml
ROOT = '/content/yolo_thermal_transformer'
DRIVE = '/content/drive/MyDrive'
DRIVE_WEIGHTS_DIR = f'{DRIVE}/thermal_weights'
if not os.path.isdir(ROOT):
    zips = glob.glob(f'{DRIVE}/transformer.zip') + glob.glob(f'{DRIVE}/*/transformer.zip')
    folders = (glob.glob(f'{DRIVE}/yolo_thermal_transformer')
               + glob.glob(f'{DRIVE}/*/yolo_thermal_transformer'))
    if zips:
        print('unzipping', zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall('/content')
    elif folders:
        print('copying', folders[0])
        shutil.copytree(folders[0], ROOT)
    else:
        raise FileNotFoundError(
            "Dataset not found. Upload 'transformer.zip' (zip of the "
            "yolo_thermal_transformer folder) anywhere in your Google Drive.")
assert os.path.isdir(f'{ROOT}/images'), f'unexpected dataset layout under {ROOT}'
p = f'{ROOT}/data_clahe.yaml'
d = yaml.safe_load(open(p)); d['path'] = ROOT
yaml.safe_dump(d, open(p, 'w'), sort_keys=False)
DATA_YAML = p
print('dataset ready ->', d)

In [ ]:
# Thermal-tuned. NO hue jitter (palette = heat). x is heavy for ~600 imgs ->
# early stopping + regularization. On OOM: MODEL='yolo26m.pt' or lower batch.
MODEL = 'yolo26x.pt'
TRAIN_ARGS = dict(data=DATA_YAML, epochs=150, imgsz=1280, batch=4, seed=1337,
                  hsv_h=0.0, hsv_s=0.2, hsv_v=0.3,
                  fliplr=0.5, flipud=0.0, degrees=10.0, translate=0.1, scale=0.5,
                  mosaic=1.0, close_mosaic=10,
                  weight_decay=0.0005, dropout=0.1, cos_lr=True, patience=30, amp=True)

In [ ]:
from ultralytics import YOLO
m = YOLO(MODEL)
m.train(project='runs/transformer', name='yolo26x', **TRAIN_ARGS)

In [ ]:
import os, shutil
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)
os.makedirs('/content/transformer_train/models', exist_ok=True)
best = str(m.trainer.best)
shutil.copy(best, f'{DRIVE_WEIGHTS_DIR}/transformer.pt')
shutil.copy(best, '/content/transformer_train/models/transformer.pt')
print('saved transformer.pt from', best)

In [ ]:
r = m.val(data=DATA_YAML, split='test')
print(f'transformer TEST: mAP50-95={r.box.map:.3f}  mAP50={r.box.map50:.3f}')

## Test the full pipeline (transformer detect + CV hotspots)

In [ ]:
import glob, os, cv2
import matplotlib.pyplot as plt
from thermal.detector import YoloDetector
from thermal.colormap import build_lut, ColorToHeat
from thermal.pipeline import analyze_image
from thermal.report import annotate, to_json

td = YoloDetector('/content/transformer_train/models/transformer.pt')
c2h = ColorToHeat(build_lut('inferno'))

imgs = sorted(glob.glob(f'{ROOT}/images/test/orig/*.jpg'))[:6]
os.makedirs('/content/cascade_out', exist_ok=True)
fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
for ax, p in zip(axes if len(imgs) > 1 else [axes], imgs):
    bgr = cv2.imread(p); rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    findings, _, calib = analyze_image(rgb, td, c2h)
    out = annotate(bgr, findings)
    cv2.imwrite(f"/content/cascade_out/{os.path.basename(p)}", out)
    print(os.path.basename(p), 'calib_ok=', calib, to_json(findings))
    ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.show()

Annotated outputs are in `/content/cascade_out/`; the transformer weights are in Drive `thermal_weights/`. Download `transformer.pt` into the repo's `models/` to run the local API. Tune hotspot sensitivity via `_HOTSPOT_MARGIN` / severity thresholds in `src/thermal/defects.py`.